# Comparativa de Modelos de Clasificación — Árbol de Decisión vs Random Forest vs XGBoost

---

**Autor:** Borja Mora Méndez
**Contacto:** [borja.mora.mendez@gmail.com](mailto:borja.mora.mendez@gmail.com) · [LinkedIn](https://www.linkedin.com/in/borja-mora-mendez/)
**Repositorio:** [Data Analytics Portfolio](https://github.com/BORJAMOME/Data-Analytics-Portfolio)
**Categoría:** Machine Learning · Supervisado · Clasificación · Comparativa de modelos

---

### Objetivo

Comparar de forma rigurosa los tres modelos de clasificación entrenados para predecir la satisfacción de clientes de un gimnasio urbano — **Árbol de Decisión**, **Random Forest** y **XGBoost** — y determinar cuál ofrece el mejor equilibrio entre rendimiento, interpretabilidad y utilidad de negocio.

### Contexto de negocio

**El cliente:** cadena de gimnasios urbanos con múltiples sedes.

**El problema:** la dirección necesita predecir qué clientes están insatisfechos para actuar antes de que se den de baja. Se han entrenado tres modelos sobre los mismos datos operativos. Este notebook evalúa cuál debería desplegarse en producción.

## 1️. Setup: librerías y configuración visual

In [ ]:
# Librerías base
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

# Preprocesamiento
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV

# Modelos
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Métricas
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve,
)
from sklearn.inspection import permutation_importance

import warnings
warnings.filterwarnings('ignore')

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", None)

# Estilo visual
PURPLE = '#7a7bff'
GREEN  = '#6E7F5B'
RED    = '#C2412E'
GRAY   = '#F4EFE6'
INK    = '#2B2118'
MUTED  = '#bfbfbf'

plt.rcParams.update({
    'figure.figsize': (10, 5),
    'figure.dpi': 100,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.edgecolor': MUTED,
    'axes.labelcolor': INK,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.titlecolor': INK,
    'xtick.color': MUTED,
    'ytick.color': MUTED,
    'font.family': 'sans-serif',
    'font.size': 10,
    'grid.color': '#f0f0f0',
    'grid.linewidth': 0.5,
})

## 2. Carga de datos y preparación

In [ ]:
data = pd.read_excel("gym_clientes.xlsx")
print(f"Registros: {data.shape[0]} | Columnas: {data.shape[1]}")
data.head()

In [ ]:
# Variables predictoras y target
X = data[["Antiguedad_Meses", "Asistencias_Mes", "Horas_Pico_Mes", "Gasto_Mensual_Extra"]]
y = data["Satisfecho"]

# Mismo split que en los notebooks individuales
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]} registros ({y_train.mean():.1%} satisfechos)")
print(f"Test:  {X_test.shape[0]} registros ({y_test.mean():.1%} satisfechos)")
print(f"\nClases equilibradas — accuracy es métrica fiable sin ajustes.")

## 3. Entrenamiento de los tres modelos

Se utilizan exactamente los mismos hiperparámetros optimizados en cada notebook individual para garantizar la comparabilidad.

In [ ]:
# 1. Árbol de Decisión (depth=2) — seleccionado por CV en el notebook individual
dt = DecisionTreeClassifier(max_depth=2, random_state=42)
dt.fit(X_train, y_train)

# 2. Random Forest (100 árboles) — configuración estándar validada por CV
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# 3. XGBoost (mejores hiperparámetros del GridSearchCV)
xgb = XGBClassifier(
    n_estimators=100, max_depth=4, learning_rate=0.05,
    eval_metric="logloss", random_state=42, use_label_encoder=False,
)
xgb.fit(X_train, y_train)

print("✓ Árbol de Decisión entrenado (max_depth=2)")
print("✓ Random Forest entrenado (n_estimators=100)")
print("✓ XGBoost entrenado (n_estimators=100, max_depth=4, lr=0.05)")

## 4. Comparación de métricas de evaluación

In [ ]:
models = {
    "Árbol de Decisión": dt,
    "Random Forest": rf,
    "XGBoost": xgb,
}

results = []
for name, model in models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    results.append({
        "Modelo": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "AUC-ROC": roc_auc_score(y_test, y_proba),
        "Errores": (y_test != y_pred).sum(),
    })

df_results = pd.DataFrame(results).set_index("Modelo")
print("Comparativa de métricas en el conjunto de Test (60 observaciones)")
print("=" * 75)
df_results.style.format({
    "Accuracy": "{:.1%}", "Precision": "{:.1%}", "Recall": "{:.1%}",
    "F1-Score": "{:.1%}", "AUC-ROC": "{:.3f}", "Errores": "{:.0f}"
})

In [ ]:
# Tabla resumen formateada
print("Comparativa de métricas — Test set (60 observaciones)")
print("─" * 75)
for _, row in df_results.iterrows():
    print(f"\n  {row.name}")
    print(f"    Accuracy:  {row['Accuracy']:.1%}")
    print(f"    Precision: {row['Precision']:.1%}")
    print(f"    Recall:    {row['Recall']:.1%}")
    print(f"    F1-Score:  {row['F1-Score']:.1%}")
    print(f"    AUC-ROC:   {row['AUC-ROC']:.3f}")
    print(f"    Errores:   {row['Errores']:.0f} / 60")

## 5. Visualización comparativa

### 5.1 Métricas lado a lado

In [ ]:
metrics = ["Accuracy", "Precision", "Recall", "F1-Score", "AUC-ROC"]
model_names = df_results.index.tolist()
colors = [GREEN, PURPLE, RED]

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(metrics))
width = 0.25

for i, (name, color) in enumerate(zip(model_names, colors)):
    values = [df_results.loc[name, m] for m in metrics]
    bars = ax.bar(x + i * width, values, width, label=name, color=color, alpha=0.85, edgecolor='white')
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{v:.1%}", ha='center', va='bottom', fontsize=8, fontweight='bold', color=INK)

ax.set_ylabel("Puntuación", fontsize=12)
ax.set_title("Comparativa de métricas — Árbol de Decisión vs Random Forest vs XGBoost",
             fontsize=13, fontweight="bold", color=INK)
ax.set_xticks(x + width)
ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylim(0.82, 1.02)
ax.legend(fontsize=10, frameon=True, facecolor='white', edgecolor=MUTED)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### 5.2 Matrices de confusión

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
cmaps = ["Greens", "Purples", "Reds"]
model_list = list(models.items())

for ax, (name, model), cmap in zip(axes, model_list, cmaps):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap,
                xticklabels=["No Satisf.", "Satisf."],
                yticklabels=["No Satisf.", "Satisf."], ax=ax,
                annot_kws={"size": 14, "weight": "bold"})
    errors = cm[0,1] + cm[1,0]
    ax.set_title(f"{name}\n({errors} errores)", fontsize=11, fontweight="bold")
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Real")

plt.suptitle("Matrices de confusión — Comparativa de los tres modelos",
             fontsize=14, fontweight="bold", color=INK, y=1.03)
plt.tight_layout()
plt.show()

### 5.3 Curvas ROC

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

line_styles = ['-', '--', '-.']
line_colors = [GREEN, PURPLE, RED]

for (name, model), ls, lc in zip(models.items(), line_styles, line_colors):
    y_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc_val = roc_auc_score(y_test, y_proba)
    ax.plot(fpr, tpr, linestyle=ls, color=lc, linewidth=2.5,
            label=f"{name} (AUC = {auc_val:.3f})")

ax.plot([0, 1], [0, 1], "k--", linewidth=1, alpha=0.5, label="Random (AUC = 0.500)")
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("Curvas ROC — Comparativa de modelos", fontsize=13, fontweight="bold", color=INK)
ax.legend(fontsize=10, loc="lower right", frameon=True, facecolor='white', edgecolor=MUTED)
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)

plt.tight_layout()
plt.show()

### 5.4 Validación cruzada comparativa

La evaluación sobre el test set puede depender de la partición concreta. La validación cruzada 5-fold ofrece una estimación más robusta del rendimiento real de cada modelo.

In [ ]:
cv_data = {}

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy")
    cv_data[name] = scores
    print(f"{name:25s} | CV Accuracy: {scores.mean():.4f} ± {scores.std():.4f}")

# Boxplot de los scores CV
fig, ax = plt.subplots(figsize=(10, 5))

positions = [1, 2, 3]
box_colors = [GREEN, PURPLE, RED]
bp = ax.boxplot(
    [cv_data[n] for n in model_names],
    positions=positions, widths=0.5, patch_artist=True,
    medianprops=dict(color=INK, linewidth=2),
    whiskerprops=dict(color=MUTED), capprops=dict(color=MUTED),
    flierprops=dict(markerfacecolor=MUTED, marker='o', markersize=6)
)

for patch, color in zip(bp['boxes'], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
    patch.set_edgecolor(INK)

# Añadir puntos individuales
for i, name in enumerate(model_names):
    scores = cv_data[name]
    x = np.random.normal(positions[i], 0.04, size=len(scores))
    ax.scatter(x, scores, alpha=0.8, color=INK, s=40, zorder=5, edgecolors='white', linewidth=0.5)

ax.set_xticklabels(model_names, fontsize=11, color=INK)
ax.set_ylabel("Accuracy (5-fold CV)", fontsize=12)
ax.set_title("Distribución del Accuracy en Validación Cruzada (5-fold)",
             fontsize=13, fontweight="bold", color=INK)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Importancia de variables — Comparativa

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
bar_colors = [GREEN, PURPLE, RED]

for ax, (name, model), color in zip(axes, models.items(), bar_colors):
    imp = pd.DataFrame({
        "feature": X.columns,
        "importance": model.feature_importances_
    }).sort_values("importance", ascending=True)
    
    bars = ax.barh(imp["feature"], imp["importance"], color=color, alpha=0.85, edgecolor='white')
    for bar, v in zip(bars, imp["importance"]):
        ax.text(v + 0.005, bar.get_y() + bar.get_height()/2, f"{v:.1%}",
                va="center", fontsize=9, color=INK)
    ax.set_title(name, fontsize=11, fontweight="bold")
    ax.set_xlim(0, 1.15)

plt.suptitle("Importancia de variables — Comparativa de modelos",
             fontsize=14, fontweight="bold", color=INK, y=1.03)
plt.tight_layout()
plt.show()

### Lectura de la importancia de variables

Los tres modelos coinciden en un hallazgo fundamental: **`Asistencias_Mes` es la variable dominante** para predecir la satisfacción de los clientes.

- **Árbol de Decisión:** concentra el **98,9%** de la importancia en `Asistencias_Mes`. Con solo 2 niveles de profundidad, el árbol no necesita más variables.
- **Random Forest:** distribuye la importancia de forma más equilibrada (51,9%), pero `Asistencias_Mes` sigue siendo la principal. La diferencia se debe a que el ensemble explora más combinaciones de variables.
- **XGBoost:** asigna el **80,8%** a `Asistencias_Mes`, situándose entre los otros dos modelos en concentración de importancia.

La convergencia de los tres modelos hacia la misma variable refuerza la robustez del hallazgo: la frecuencia de asistencia es, con diferencia, el indicador más potente de satisfacción.

## 7. Tabla resumen — ¿Qué modelo elegir?

In [ ]:
# Tabla resumen con todas las dimensiones de comparación
summary = pd.DataFrame({
    "Modelo": ["Árbol de Decisión", "Random Forest", "XGBoost"],
    "Accuracy": ["90,0%", "90,0%", "91,7%"],
    "Recall": ["89,7%", "89,7%", "93,1%"],
    "AUC-ROC": ["0,909", "0,917", "0,917"],
    "Errores (de 60)": [6, 6, 5],
    "Interpretabilidad": ["⭐⭐⭐ Alta", "⭐⭐ Media", "⭐ Baja"],
    "Complejidad": ["Baja", "Media", "Alta"],
    "Tiempo de tuning": ["Mínimo", "Bajo", "Alto (GridSearch)"],
    "Variable clave": ["Asistencias (98,9%)", "Asistencias (51,9%)", "Asistencias (80,8%)"],
}).set_index("Modelo")

print("TABLA RESUMEN — Comparativa de modelos")
print("=" * 90)
for col in summary.columns:
    print(f"\n{col}:")
    for idx in summary.index:
        print(f"  {idx:25s} → {summary.loc[idx, col]}")

### Perfil de cada modelo — Radar chart

In [ ]:
from matplotlib.patches import FancyBboxPatch

# Métricas normalizadas para el radar
categories = ["Accuracy", "Recall", "AUC-ROC", "Interpretabilidad", "Simplicidad"]

dt_vals = [0.900, 0.897, 0.909, 1.0, 1.0]
rf_vals = [0.900, 0.897, 0.917, 0.6, 0.6]
xgb_vals = [0.917, 0.931, 0.917, 0.3, 0.3]

angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
angles += angles[:1]

dt_vals += dt_vals[:1]
rf_vals += rf_vals[:1]
xgb_vals += xgb_vals[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

ax.plot(angles, dt_vals, 'o-', linewidth=2, color=GREEN, label='Árbol de Decisión', markersize=6)
ax.fill(angles, dt_vals, alpha=0.1, color=GREEN)

ax.plot(angles, rf_vals, 's-', linewidth=2, color=PURPLE, label='Random Forest', markersize=6)
ax.fill(angles, rf_vals, alpha=0.1, color=PURPLE)

ax.plot(angles, xgb_vals, 'D-', linewidth=2, color=RED, label='XGBoost', markersize=6)
ax.fill(angles, xgb_vals, alpha=0.1, color=RED)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11, color=INK)
ax.set_ylim(0, 1.05)
ax.set_title("Perfil comparativo de los tres modelos",
             fontsize=14, fontweight="bold", color=INK, pad=25)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=10,
          frameon=True, facecolor='white', edgecolor=MUTED)

ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(["0.2", "0.4", "0.6", "0.8", "1.0"], fontsize=8, color=MUTED)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. ¿La diferencia es estadísticamente significativa?

XGBoost obtiene un 1,7% más de accuracy que los otros dos modelos. Pero con solo 60 observaciones en test, ¿es una diferencia real o ruido estadístico?

In [ ]:
from scipy import stats

# Test de McNemar entre pares de modelos
y_pred_dt = dt.predict(X_test)
y_pred_rf = rf.predict(X_test)
y_pred_xgb = xgb.predict(X_test)

def mcnemar_test(y_true, pred_a, pred_b, name_a, name_b):
    correct_a = (pred_a == y_true)
    correct_b = (pred_b == y_true)
    
    # Tabla de contingencia
    b = ((correct_a) & (~correct_b)).sum()  # A acierta, B falla
    c = ((~correct_a) & (correct_b)).sum()  # A falla, B acierta
    
    # Test exacto binomial (mejor para muestras pequeñas)
    n = b + c
    if n == 0:
        p_value = 1.0
    else:
        p_value = stats.binom_test(min(b, c), n, 0.5)
    
    print(f"  {name_a} vs {name_b}")
    print(f"    Solo {name_a} acierta: {b} casos")
    print(f"    Solo {name_b} acierta: {c} casos")
    print(f"    p-valor: {p_value:.4f} {'← significativo' if p_value < 0.05 else '← NO significativo'}")
    print()

print("Test de McNemar (binomial exacto) — ¿Los modelos cometen errores diferentes?")
print("─" * 70)
mcnemar_test(y_test, y_pred_dt, y_pred_rf, "Árbol", "Random Forest")
mcnemar_test(y_test, y_pred_dt, y_pred_xgb, "Árbol", "XGBoost")
mcnemar_test(y_test, y_pred_rf, y_pred_xgb, "Random Forest", "XGBoost")

print("Conclusión: con 60 observaciones, las diferencias entre modelos NO son")
print("estadísticamente significativas. Los tres modelos ofrecen un rendimiento")
print("equivalente en la práctica.")

## 9. Veredicto final — ¿Cuál es el mejor modelo?

### Resumen de hallazgos

| Dimensión | Árbol de Decisión | Random Forest | XGBoost |
|---|---|---|---|
| **Accuracy** | 90,0% | 90,0% | **91,7%** |
| **Recall** | 89,7% | 89,7% | **93,1%** |
| **AUC-ROC** | 0,909 | **0,917** | **0,917** |
| **Errores** | 6 | 6 | **5** |
| **Interpretabilidad** | **⭐⭐⭐** | ⭐⭐ | ⭐ |
| **Complejidad** | **Baja** | Media | Alta |
| **Variable clave** | Asistencias (98,9%) | Asistencias (51,9%) | Asistencias (80,8%) |

### El veredicto

> **Para este problema concreto, el Árbol de Decisión (`max_depth=2`) es el modelo recomendado.**

#### ¿Por qué no XGBoost, si tiene mejor accuracy?

1. **La diferencia no es significativa.** XGBoost acierta 1 predicción más de 60 (91,7% vs 90,0%). El test de McNemar confirma que esta diferencia no es estadísticamente significativa con el tamaño muestral disponible.

2. **La señal es demasiado clara para un modelo complejo.** Los tres modelos coinciden en que `Asistencias_Mes` es, con enorme diferencia, la variable más importante. Cuando la señal está tan concentrada en una sola variable, la complejidad adicional de un ensemble no aporta valor predictivo real.

3. **El Árbol de Decisión es completamente interpretable.** Sus dos reglas pueden explicarse a cualquier director en 30 segundos:
   - *"Si el cliente asiste más de 13 veces al mes y lleva más de 2 meses, está satisfecho."*
   - Esa transparencia tiene un valor enorme en contextos de negocio donde las decisiones deben justificarse.

4. **Menos complejidad = menos riesgo operativo.** Un árbol de 2 reglas puede implementarse como un filtro en el CRM sin necesidad de infraestructura de ML. No requiere mantenimiento, no tiene dependencias y no falla en silencio.

### ¿Cuándo elegir Random Forest o XGBoost?

- **Random Forest** sería preferible si el dataset tuviera más variables con señales independientes, o si necesitáramos estimar la incertidumbre de las predicciones.
- **XGBoost** sería la elección correcta con datasets más grandes, señales más sutiles o cuando cada décima de accuracy tiene un impacto económico medible (fraude, pricing dinámico, etc.).

### Recomendación de negocio

El gimnasio no necesita desplegar un modelo de Machine Learning en producción. La regla del árbol de decisión puede convertirse directamente en una **alerta operativa en el CRM**:

> **"Si un cliente registra menos de 14 asistencias en el último mes → incluir en programa de retención."**

Esta regla captura el 90% de la señal predictiva con coste de implementación prácticamente nulo.